# LangChain RAG Pipeline

Walkthrough of the same pipeline in `Interview-prep/main.py` — broken into cells so each step can be run and inspected independently.

**RAG = Retrieval-Augmented Generation**
1. **(Offline / Ingest)** Embed your docs → store in Pinecone
2. **(Online / Ask)** Embed the question → fetch similar chunks → inject into prompt → LLM answers

**Stack:** Groq `llama-3.1-8b-instant` · Pinecone `llama-text-embed-v2` · LangChain LCEL

## 0. Install Dependencies

Install all required packages into the project virtualenv. Run this cell once — `uv add` is idempotent.

In [ ]:
!uv add langchain-groq langchain-pinecone langchain-text-splitters pinecone python-dotenv httpx ragas datasets

## 1. Setup

Load API keys from `AgenticAI/.env`. Both `GROQ_API_KEY` and `PINECONE_API_KEY` must be set before any other cell runs.

In [ ]:
import os
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Load GROQ_API_KEY and PINECONE_API_KEY from AgenticAI/.env
load_dotenv(Path("../.env"))

print("GROQ_API_KEY:", "set" if os.environ.get("GROQ_API_KEY") else "MISSING")
print("PINECONE_API_KEY:", "set" if os.environ.get("PINECONE_API_KEY") else "MISSING")

## 2. Primitive: ChatModel (the LLM)

`ChatGroq` wraps Groq's API as a LangChain `ChatModel` — a Runnable that accepts messages and returns an `AIMessage`.

`temperature=0` makes responses deterministic (no creative variation — good for RAG).

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"],
    # Force IPv4 — avoids ~24s IPv6 stall on this host
    http_client=httpx.Client(
        transport=httpx.HTTPTransport(local_address="0.0.0.0"),
        timeout=60.0,
    ),
)

# Quick smoke test
response = llm.invoke("Say hello in one word.")
print(response.content)

## 3. Primitive: Embeddings

`PineconeEmbeddings` converts text → dense float vectors (1024 dimensions).  
Semantically similar text lands close together in vector space — this enables similarity search.

Pinecone hosts the model, so no local GPU needed.

In [ ]:
from langchain_pinecone import PineconeEmbeddings

embeddings = PineconeEmbeddings(
    model="llama-text-embed-v2",
    pinecone_api_key=os.environ["PINECONE_API_KEY"],
)

# Inspect a vector
vector = embeddings.embed_query("Python experience")
print(f"Dimensions: {len(vector)}")
print(f"First 5 values: {vector[:5]}")

## 4. Primitive: Pinecone Index

Create the index on first run — idempotent after that.

- `dimension=1024` must match the embedding model output
- `metric=cosine` measures angle between vectors — best for semantic text similarity

In [ ]:
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = "interview-prep"

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

existing = {i.name for i in pc.list_indexes()}
print(f"Existing indexes: {existing}")

if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index: {INDEX_NAME}")
else:
    print(f"Index already exists: {INDEX_NAME}")

## 5. Primitive: Document

`Document` is LangChain's universal text container:
```python
Document(
    page_content = "the raw text",
    metadata     = {"source": "rag.md"}
)
```
Every downstream component — splitter, vector store, prompt — speaks `Document`.

In [ ]:
from langchain_core.documents import Document

DOCS_DIR = Path("../data/documents")  # AgenticAI/data/documents/

docs = []
for path in DOCS_DIR.rglob("*"):
    if path.suffix in {".txt", ".md"} and path.is_file():
        docs.append(Document(
            page_content=path.read_text(),
            metadata={"source": path.name},
        ))

print(f"Loaded {len(docs)} document(s)")
for d in docs:
    print(f"  {d.metadata['source']} — {len(d.page_content)} chars")

## 6. Primitive: TextSplitter

`RecursiveCharacterTextSplitter` splits docs into chunks that fit the LLM's context window.

**Why split?** Retrieval is more precise on small focused chunks than whole documents.

**Why `chunk_overlap=200`?** Prevents losing context at chunk boundaries — sentences split across edges still make sense.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = splitter.split_documents(docs)
print(f"{len(docs)} doc(s) → {len(chunks)} chunk(s)")

# Inspect first chunk
if chunks:
    print(f"\nFirst chunk ({len(chunks[0].page_content)} chars):")
    print(chunks[0].page_content[:300], "...")

## 7. Ingest Pipeline: Embed → Store

`PineconeVectorStore.from_documents()` embeds every chunk and upserts into Pinecone in one call.

Each chunk becomes one vector in the index, keyed by an auto-generated ID.

In [ ]:
from langchain_pinecone import PineconeVectorStore

if not chunks:
    print(f"No docs found in {DOCS_DIR} — skipping ingest")
else:
    PineconeVectorStore.from_documents(chunks, embeddings, index_name=INDEX_NAME)
    print(f"Ingested {len(chunks)} chunks into '{INDEX_NAME}'")

## 8. Query Pipeline (LCEL RAG Chain)

**Retrieve → Augment → Generate**

LCEL (`|` pipe) composes Runnables into a chain. Every LangChain object implements `.invoke(input)`:

```
prompt | llm
  └─ prompt.invoke({context, question}) → formatted messages
       └─ llm.invoke(messages) → AIMessage
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

QUESTION = "What is my experience with Python?"  # change this

# ── STEP 1: RETRIEVE ──────────────────────────────────────────────────────────
store = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)
retrieved_docs = store.as_retriever(search_kwargs={"k": 3}).invoke(QUESTION)

print(f"Retrieved {len(retrieved_docs)} chunk(s)")
for i, d in enumerate(retrieved_docs):
    print(f"\n[{i+1}] {d.metadata.get('source', '?')}")
    print(d.page_content[:200], "...")

In [ ]:
# ── STEP 2: AUGMENT ───────────────────────────────────────────────────────────
context = "\n\n".join(d.page_content for d in retrieved_docs)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an interview coach. Answer the question using only the "
        "provided context. Be specific and concise. If the context doesn't "
        "contain the answer, say so.",
    ),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

# ── STEP 3: GENERATE ──────────────────────────────────────────────────────────
chain = prompt | llm
response = chain.invoke({"context": context, "question": QUESTION})

print(f"Q: {QUESTION}")
print(f"A: {response.content}")

## 9. RAGAS Evaluation

RAGAS measures RAG pipeline quality across four metrics:

| Metric | What it measures |
|--------|-----------------|
| **Faithfulness** | Is the answer grounded in the retrieved context? (no hallucination) |
| **Answer Relevancy** | Does the answer actually address the question? |
| **Context Precision** | Are the retrieved chunks relevant to the question? |
| **Context Recall** | Did retrieval find everything needed to answer? |

All scores are 0–1. Higher is better.

In [ ]:
# Test dataset — questions grounded in AgenticAI/data/documents/
# ground_truth: reference answer used by context_recall scoring

TEST_DATASET = [
    {
        "question": "What embedding model does Pinecone host and what are its dimensions?",
        "ground_truth": "llama-text-embed-v2 is a 1024-dimensional embedding model hosted by Pinecone.",
    },
    {
        "question": "What is Groq and what makes it distinctive?",
        "ground_truth": "Groq serves open models like Llama 3.1 with very low latency.",
    },
    {
        "question": "How does LangChain compose LLM calls?",
        "ground_truth": "LangChain composes LLM calls into linear chains using the pipe operator (LCEL).",
    },
    {
        "question": "How does LangGraph differ from LangChain?",
        "ground_truth": "LangGraph adds stateful, graph-based control flow on top of LangChain, enabling branches, loops, and persistence.",
    },
    {
        "question": "What is Pinecone used for?",
        "ground_truth": "Pinecone is a managed serverless vector database for similarity search over embeddings.",
    },
    {
        "question": "What is Retrieval-Augmented Generation?",
        "ground_truth": "Retrieval-Augmented Generation (RAG) grounds a model's answers in retrieved documents.",
    },
]

print(f"{len(TEST_DATASET)} test cases loaded")

In [ ]:
# Run the RAG pipeline on every test question to collect answers + contexts

store = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)
retriever = store.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer the question using only the provided context. "
        "Be specific and concise. If the context doesn't contain the answer, say so.",
    ),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])
chain = prompt | llm

results = []
for item in TEST_DATASET:
    q = item["question"]
    retrieved = retriever.invoke(q)
    ctx = "\n\n".join(d.page_content for d in retrieved)
    answer = chain.invoke({"context": ctx, "question": q}).content
    results.append({
        "question": q,
        "answer": answer,
        "contexts": [d.page_content for d in retrieved],
        "ground_truth": item["ground_truth"],
    })
    print(f"✓ {q[:70]}")

print(f"\nCollected {len(results)} results")

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

dataset = Dataset.from_list(results)

scores = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=llm,
    embeddings=embeddings,
)

scores.to_pandas()

In [ ]:
# Summary — mean score per metric
df = scores.to_pandas()
print("=== RAGAS Summary ===")
for col in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    print(f"{col:25s}: {df[col].mean():.3f}")